# Chapter 20 — Procedural Memory: Updating the Agent's Own Instructions

Hands-On LangChain. Code targets **LangGraph 1.x** + **LangMem**.

We've given our email assistant **semantic** memory (facts) and **episodic** memory (few-shot examples).
The final pillar is **procedural memory**: the agent's own **instructions and rules**. Until now these
were hardcoded. In this chapter we move them into the long-term **store** so they can be **updated** —
both directly and, more powerfully, **by an LLM** that turns user feedback into improved instructions.

This lets the agent **learn from feedback**, **adapt to changing needs**, and **stay consistent** (the
updated instructions persist).

Companion notebook for Chapter 20. Reuses `prompts.py`.

```bash
pip install langgraph langmem langchain langchain-openai python-dotenv -q
```

## 20.1 Setting up the email assistant

We re-establish the profile, rules, store, and the episodic-memory helpers from Chapter 19.

In [1]:
import os, warnings, uuid
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv(usecwd=True))
warnings.filterwarnings("ignore")

profile = {
    "name": "Youssef", "full_name": "Youssef Hosni",
    "user_profile_background": "Senior Data Scientist leading a team of 5 developers.",
}
prompt_instructions = {
    "triage_rules": {
        "ignore": "Marketing newsletters, spam emails, mass company announcements.",
        "notify": "Team member out sick, build system notifications, project status updates.",
        "respond": "Direct questions from team members, meeting requests, critical bug reports.",
    },
    "agent_instructions": "Use these tools when appropriate to help manage Youssef's tasks efficiently.",
}

from langgraph.store.memory import InMemoryStore
store = InMemoryStore(index={"embed": f"openai:{os.getenv('EMBEDDING_MODEL', 'text-embedding-3-small')}"})

# episodic-memory formatter (from Ch19)
_tmpl = "Email Subject: {subject}\nEmail From: {from_email}\n> Triage Result: {result}"
def format_few_shot_examples(examples):
    strs = ["Here are some previous examples:"]
    for eg in examples:
        strs.append(_tmpl.format(subject=eg.value["email"]["subject"],
                                 from_email=eg.value["email"]["author"], result=eg.value["label"]))
    return "\n\n------------\n\n".join(strs)

## 20.2 A dynamic triage router with procedural memory

The key change is in `triage_router`: instead of using the hardcoded `prompt_instructions`, it **fetches
each triage rule from the store** (keyed per user). If a rule isn't there yet, it seeds the store with
the default — so the agent always has a starting set of rules that can later be changed.

In [2]:
import warnings; warnings.filterwarnings("ignore")
from pydantic import BaseModel, Field
from typing_extensions import TypedDict, Annotated
from typing import Literal
from langchain.chat_models import init_chat_model
from prompts import triage_system_prompt, triage_user_prompt
from langgraph.graph import add_messages, StateGraph, START, END
from langgraph.types import Command
from langgraph.store.base import BaseStore
from langchain_core.runnables import RunnableConfig

llm = init_chat_model(f"openai:{os.getenv('MODEL', 'gpt-5.4-mini')}", temperature=0)

class Router(BaseModel):
    reasoning: str = Field(description="Step-by-step reasoning behind the classification.")
    classification: Literal["ignore", "respond", "notify"] = Field(description="'ignore', 'notify', or 'respond'.")
llm_router = llm.with_structured_output(Router)

class State(TypedDict):
    email_input: dict
    messages: Annotated[list, add_messages]

def get_or_seed(store, namespace, key, default):
    """Fetch a prompt from the store, seeding it with a default the first time."""
    result = store.get(namespace, key)
    if result is None:
        store.put(namespace, key, {"prompt": default})
        return default
    return result.value["prompt"]

def triage_router(state: State, config: RunnableConfig, store: BaseStore) -> Command[Literal["response_agent", "__end__"]]:
    e = state["email_input"]
    user_id = config["configurable"]["langgraph_user_id"]

    # episodic memory (few-shot examples)
    examples = store.search(("email_assistant", user_id, "examples"), query=str({"email": e}))
    examples = format_few_shot_examples(examples)

    # procedural memory (triage rules) — fetched from the store, seeded with defaults
    ns = (user_id,)
    ignore_prompt  = get_or_seed(store, ns, "triage_ignore",  prompt_instructions["triage_rules"]["ignore"])
    notify_prompt  = get_or_seed(store, ns, "triage_notify",  prompt_instructions["triage_rules"]["notify"])
    respond_prompt = get_or_seed(store, ns, "triage_respond", prompt_instructions["triage_rules"]["respond"])

    system_prompt = triage_system_prompt.format(
        full_name=profile["full_name"], name=profile["name"],
        user_profile_background=profile["user_profile_background"],
        triage_no=ignore_prompt, triage_notify=notify_prompt, triage_email=respond_prompt,
        examples=examples,
    )
    user_prompt = triage_user_prompt.format(
        author=e["author"], to=e["to"], subject=e["subject"], email_thread=e["email_thread"])
    result = llm_router.invoke([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ])
    if result.classification == "respond":
        print("📧 Classification: RESPOND - This email requires a response")
        return Command(goto="response_agent",
                       update={"messages": [{"role": "user", "content": f"Respond to the email {e}"}]})
    elif result.classification == "ignore":
        print("🚫 Classification: IGNORE - This email can be safely ignored")
        return Command(goto=END)
    else:
        print("🔔 Classification: NOTIFY - This email contains important information")
        return Command(goto=END)

## 20.3 A response agent with dynamic instructions

The response agent's `create_prompt` is updated the same way: it **fetches the agent instructions from
the store** (seeding the default the first time), so they too become procedural memory we can update.

In [3]:
from langchain_core.tools import tool
from langmem import create_manage_memory_tool, create_search_memory_tool
from langgraph.prebuilt import create_react_agent

@tool
def write_email(to: str, subject: str, content: str) -> str:
    """Write and send an email."""
    return f"Email sent to {to} with subject '{subject}'"

@tool
def schedule_meeting(attendees: list[str], subject: str, duration_minutes: int, preferred_day: str) -> str:
    """Schedule a calendar meeting."""
    return f"Meeting '{subject}' scheduled for {preferred_day} with {len(attendees)} attendees"

@tool
def check_calendar_availability(day: str) -> str:
    """Check calendar availability for a given day."""
    return f"Available times on {day}: 9:00 AM, 2:00 PM, 4:00 PM"

manage_memory_tool = create_manage_memory_tool(namespace=("email_assistant", "{langgraph_user_id}", "collection"))
search_memory_tool = create_search_memory_tool(namespace=("email_assistant", "{langgraph_user_id}", "collection"))

agent_system_prompt_memory = """
< Role > You are {full_name}'s executive assistant. </ Role >
< Tools > write_email, schedule_meeting, check_calendar_availability, manage_memory, search_memory </ Tools >
< Instructions > {instructions} </ Instructions >
"""

def create_prompt(state, config, store):
    user_id = config["configurable"]["langgraph_user_id"]
    instructions = get_or_seed(store, (user_id,), "agent_instructions",
                               prompt_instructions["agent_instructions"])
    return [{"role": "system",
             "content": agent_system_prompt_memory.format(instructions=instructions, **profile)}] + state["messages"]

tools = [write_email, schedule_meeting, check_calendar_availability, manage_memory_tool, search_memory_tool]
response_agent = create_react_agent(f"openai:{os.getenv('MODEL', 'gpt-5.4-mini')}",
                                    tools=tools, prompt=create_prompt, store=store)

email_agent = StateGraph(State)
email_agent.add_node("triage_router", triage_router)
email_agent.add_node("response_agent", response_agent)
email_agent.add_edge(START, "triage_router")
email_agent = email_agent.compile(store=store)

## 20.4 Demonstrating dynamic instructions

Run the agent on an urgent email. (This also seeds the store with the default instructions on first use.)

In [4]:
email_input = {
    "author": "Alice Jones <alice.jones@bar.com>",
    "to": "Youssef Hosni <Youssef.Hosni@ToDataBeyond.com>",
    "subject": "Quick question about API documentation",
    "email_thread": "Hi Youssef, Urgent issue - your service is down. Is there a reason why?",
}
config = {"configurable": {"langgraph_user_id": "lance"}}
response = email_agent.invoke({"email_input": email_input}, config=config)
print(response["messages"][-1].content[:300])

📧 Classification: RESPOND - This email requires a response


Done — I replied to Alice asking for details so the issue can be investigated quickly.


Let's inspect the current instructions stored for this user — these are the defaults, seeded automatically:

In [5]:
print("agent_instructions:", store.get(("lance",), "agent_instructions").value["prompt"])
print("triage_respond     :", store.get(("lance",), "triage_respond").value["prompt"])

agent_instructions: Use these tools when appropriate to help manage Youssef's tasks efficiently.
triage_respond     : Direct questions from team members, meeting requests, critical bug reports.


## 20.5 Using an LLM to update instructions

Now the powerful part: using an LLM to **rewrite the instructions from feedback**. LangMem's
**`create_multi_prompt_optimizer`** takes the conversation *trajectory* plus a piece of user *feedback*,
decides which prompts to change, and how.

Here the feedback is "always sign your emails `Youssef Hosni`".

In [6]:
from langmem import create_multi_prompt_optimizer
import json

conversations = [(response["messages"], "Always sign your emails `Youssef Hosni`")]

prompts = [
    {"name": "main_agent",
     "prompt": store.get(("lance",), "agent_instructions").value["prompt"],
     "update_instructions": "keep the instructions short and to the point",
     "when_to_update": "Update whenever there is feedback on how the agent should write emails or schedule events"},
    {"name": "triage-ignore",
     "prompt": store.get(("lance",), "triage_ignore").value["prompt"],
     "update_instructions": "keep the instructions short and to the point",
     "when_to_update": "Update whenever there is feedback on which emails should be ignored"},
    {"name": "triage-notify",
     "prompt": store.get(("lance",), "triage_notify").value["prompt"],
     "update_instructions": "keep the instructions short and to the point",
     "when_to_update": "Update whenever there is feedback on which emails the user should be notified of"},
    {"name": "triage-respond",
     "prompt": store.get(("lance",), "triage_respond").value["prompt"],
     "update_instructions": "keep the instructions short and to the point",
     "when_to_update": "Update whenever there is feedback on which emails should be responded to"},
]

optimizer = create_multi_prompt_optimizer(f"openai:{os.getenv('MODEL', 'gpt-5.4-mini')}", kind="prompt_memory")
updated = optimizer.invoke({"trajectories": conversations, "prompts": prompts})
print(json.dumps(updated[0], indent=2))

{
  "name": "main_agent",
  "prompt": "Use these tools when appropriate to help manage Youssef's tasks efficiently. Always sign emails with `Youssef Hosni`.",
  "update_instructions": "keep the instructions short and to the point",
  "when_to_update": "Update whenever there is feedback on how the agent should write emails or schedule events"
}


The optimizer rewrote the **main_agent** prompt to include the signing instruction. Now we persist any changed prompts back to the store.

In [7]:
KEY = {"main_agent": "agent_instructions", "triage-ignore": "triage_ignore",
       "triage-notify": "triage_notify", "triage-respond": "triage_respond"}
for old, new in zip(prompts, updated):
    if new["prompt"] != old["prompt"]:
        print("updated", old["name"])
        store.put(("lance",), KEY[old["name"]], {"prompt": new["prompt"]})

print("\nstored agent_instructions now:")
print(store.get(("lance",), "agent_instructions").value["prompt"])

updated main_agent



stored agent_instructions now:
Use these tools when appropriate to help manage Youssef's tasks efficiently. Always sign emails with `Youssef Hosni`.


Re-run the **same** email. Because the agent now fetches its (updated) instructions from the store, it should sign the email as requested.

In [8]:
response = email_agent.invoke({"email_input": email_input}, config=config)
for m in response["messages"]:
    m.pretty_print()

📧 Classification: RESPOND - This email requires a response


================================ Human Message =================================

Respond to the email {'author': 'Alice Jones <alice.jones@bar.com>', 'to': 'Youssef Hosni <Youssef.Hosni@ToDataBeyond.com>', 'subject': 'Quick question about API documentation', 'email_thread': 'Hi Youssef, Urgent issue - your service is down. Is there a reason why?'}
================================== Ai Message ==================================
Tool Calls:
  write_email (call_tsoitCCbgdM7cEGFvjPfcBnp)
 Call ID: call_tsoitCCbgdM7cEGFvjPfcBnp
  Args:
    to: Alice Jones <alice.jones@bar.com>
    subject: Re: Quick question about API documentation
    content: Hi Alice,

Thanks for reaching out. I’m sorry for the trouble, but the note in the thread mentions the service being down rather than API documentation.

If you’re asking about the API docs, I’m happy to help—please send over the specific question or the section you’re referring to. If your concern is the service outage, please share any details you

The agent now signs off as **"Youssef Hosni"** — a direct result of the procedural-memory update. We
changed the agent's behavior without touching the code: we changed its *instructions*, stored in memory.

> 🖼️ **Image suggestion — `images/ch20/Figure 1.png`**
> A procedural-memory loop: the agent's **instructions** live in the **store** (per user). 1) The agent
> runs using the stored instructions. 2) The user gives **feedback** ("always sign as Youssef Hosni").
> 3) An **LLM optimizer** rewrites the instructions from the feedback. 4) The new instructions are saved
> back to the store, so the next run behaves differently. Emphasize the closed loop: behavior changes by
> updating *instructions in memory*, not code.

## 20.6 Another example: updating a triage rule

The same mechanism can change *triage* behavior. Let's give feedback to **ignore emails from Alice
Jones**, optimize, persist, and re-run.

In [9]:
conversations = [(response["messages"], "Ignore any emails from Alice Jones.")]
prompts = [
    {"name": "main_agent", "prompt": store.get(("lance",), "agent_instructions").value["prompt"],
     "update_instructions": "keep it short", "when_to_update": "feedback on writing emails"},
    {"name": "triage-ignore", "prompt": store.get(("lance",), "triage_ignore").value["prompt"],
     "update_instructions": "keep it short", "when_to_update": "feedback on which emails to ignore"},
    {"name": "triage-notify", "prompt": store.get(("lance",), "triage_notify").value["prompt"],
     "update_instructions": "keep it short", "when_to_update": "feedback on which emails to notify"},
    {"name": "triage-respond", "prompt": store.get(("lance",), "triage_respond").value["prompt"],
     "update_instructions": "keep it short", "when_to_update": "feedback on which emails to respond to"},
]
updated = optimizer.invoke({"trajectories": conversations, "prompts": prompts})
for old, new in zip(prompts, updated):
    if new["prompt"] != old["prompt"]:
        print("updated", old["name"])
        store.put(("lance",), KEY[old["name"]], {"prompt": new["prompt"]})
print("\ntriage_ignore now:", store.get(("lance",), "triage_ignore").value["prompt"])

updated triage-ignore


updated triage-respond



triage_ignore now: Marketing newsletters, spam emails, mass company announcements. Ignore any emails from Alice Jones.


Now re-run the same email from Alice Jones — the updated ignore rule should change the triage result.

In [10]:
_ = email_agent.invoke({"email_input": email_input}, config=config)

🚫 Classification: IGNORE - This email can be safely ignored


The agent now classifies the email from Alice Jones as **IGNORE** — its triage rule was rewritten
from feedback and persisted.

This completes our tour of **agentic memory**. Our email assistant now has all three pillars working
together:

- **Semantic memory** — facts about the user and contacts (Chapter 18).
- **Episodic memory** — few-shot examples that personalize triage (Chapter 19).
- **Procedural memory** — instructions and rules the agent updates from feedback (this chapter).

Together they turn a static agent into one that genuinely **learns, adapts, and personalizes** over time.